# Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)

# Visualization settings
sns.set_theme(style="whitegrid")

# Load the Datasets

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# Load datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Preserve test passenger IDs for submission
test_passenger_ids = test_df['PassengerId'].copy()

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

# Initial Data Exploration

In [ ]:
#Preview the Training Dataset
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

###Target Variable Distribution

In [ ]:
train_df['Survived'].value_counts()

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(data=train_df, x='Survived')

plt.title('Distribution of Survival')
plt.xlabel('Survived (0 = No, 1 = Yes)')
plt.ylabel('Number of Passengers')

plt.show()

# Exploratory Data Analysis (EDA)

###Missing Values

In [ ]:
print("Missing values in training data:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

print("\nMissing values in testing data:")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])


In [ ]:
missing_values = train_df.isnull().sum()
missing_values = missing_values[missing_values > 0]

missing_values.sort_values(ascending=False).plot(
    kind='bar',
    figsize=(8, 4)
)

plt.title('Missing Values per Column')
plt.xlabel('Features')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=45)

plt.show()

###Categorical Features

In [ ]:
categorical_cols = ['Sex', 'Pclass', 'Embarked']

for col in categorical_cols:
    print(f"\nDistribution of {col}:")
    print(train_df[col].value_counts())

###Survival Analysis by Sex

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(data=train_df, x='Sex', hue='Survived')

plt.title('Survival Distribution by Sex')
plt.xlabel('Sex')
plt.ylabel('Number of Passengers')

plt.show()

In [ ]:
survival_by_sex = train_df.groupby('Sex')['Survived'].mean() * 100

print(survival_by_sex.round(2))

###Survival Analysis by Passenger Class

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(data=train_df, x='Pclass', hue='Survived')

plt.title('Survival Distribution by Passenger Class')
plt.xlabel('Passenger Class')
plt.ylabel('Number of Passengers')

plt.show()

In [ ]:
survival_by_class = train_df.groupby('Pclass')['Survived'].mean() * 100

print(survival_by_class.round(2))

###Age Analysis

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=train_df,
    x='Age',
    hue='Survived',
    bins=30,
    kde=True,
    multiple='stack'
)

plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.ylabel('Number of Passengers')

plt.show()

In [ ]:
train_df.groupby('Survived')['Age'].describe()

###Fare Analysis

In [ ]:
train_df.groupby('Survived')['Fare'].describe()

###Family Analysis

In [ ]:
train_df['FamilySize'] = (
    train_df['SibSp'] + train_df['Parch'] + 1
)

In [ ]:
family_analysis = train_df.groupby('FamilySize').agg(
    Passenger_Count=('Survived', 'count'),
    Survivors=('Survived', 'sum'),
    Survival_Rate=('Survived', 'mean')
)

family_analysis['Survival_Rate'] *= 100

family_analysis.round(2)

In [ ]:
train_df['IsAlone'] = (
    train_df['FamilySize'] == 1
).astype(int)

In [ ]:
survival_by_alone = (
    train_df.groupby('IsAlone')['Survived'].mean() * 100
)

print(survival_by_alone.round(2))

###Title Analysis

In [ ]:
train_df['Title'] = train_df['Name'].str.extract(r',\s*([^.]*)\.')

train_df['Title'].value_counts()

In [ ]:
# Normalize equivalent titles
train_df['Title'] = train_df['Title'].replace({
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'Mme': 'Mrs'
})

# Group rare titles
rare_titles = [
    'Dr', 'Rev', 'Col', 'Major', 'Don',
    'Lady', 'Sir', 'Capt', 'the Countess', 'Jonkheer'
]

train_df['Title'] = train_df['Title'].replace(
    rare_titles, 'Rare'
)

train_df['Title'].value_counts()

In [ ]:
survival_by_title = train_df.groupby('Title')['Survived'].agg(
    ['count', 'sum', 'mean']
)

survival_by_title['Survival_Rate'] = (
    survival_by_title['mean'] * 100
)

survival_by_title.drop(columns='mean').sort_values(
    by='Survival_Rate',
    ascending=False
).round(2)

###Cabin Deck Analysis

In [ ]:
train_df['Deck'] = train_df['Cabin'].str[0]

train_df['Deck'].value_counts(dropna=False)

In [ ]:
train_df['Deck'] = train_df['Deck'].fillna('Unknown')

# Group the rare T deck category
train_df['Deck'] = train_df['Deck'].replace('T', 'Rare')

train_df['Deck'].value_counts()

In [ ]:
survival_by_deck = train_df.groupby('Deck')['Survived'].agg(
    ['count', 'sum', 'mean']
)

survival_by_deck['Survival_Rate'] = (
    survival_by_deck['mean'] * 100
)

survival_by_deck.drop(columns='mean').sort_values(
    by='Survival_Rate',
    ascending=False
).round(2)

###Ticket Group Analysis

In [ ]:
train_df['Ticket'].value_counts().head(15)

In [ ]:
train_df['TicketGroupSize'] = train_df.groupby('Ticket')['Ticket'].transform('count')

In [ ]:
train_df['TicketGroupSize'].value_counts().sort_index()

In [ ]:
survival_by_ticket = train_df.groupby('TicketGroupSize')['Survived'].agg(
    ['count', 'sum', 'mean']
)

survival_by_ticket['Survival_Rate'] = (
    survival_by_ticket['mean'] * 100
)

survival_by_ticket.drop(columns='mean').round(2)

# Feature Engineering

In [ ]:
def feature_engineering(df):

    df = df.copy()

    # 1. Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # 2. Is alone
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # 3. Extract passenger title
    df['Title'] = df['Name'].str.extract(r',\s*([^.]*)\.')

    # Normalize equivalent titles
    df['Title'] = df['Title'].replace({
        'Mlle': 'Miss',
        'Ms': 'Miss',
        'Mme': 'Mrs'
    })

    # Group rare titles
    rare_titles = [
        'Dr', 'Rev', 'Col', 'Major', 'Don',
        'Lady', 'Sir', 'Capt', 'the Countess', 'Jonkheer'
    ]

    df['Title'] = df['Title'].replace(rare_titles, 'Rare')

    # 4. Extract cabin deck
    df['Deck'] = df['Cabin'].str[0].fillna('Unknown')
    df['Deck'] = df['Deck'].replace('T', 'Rare')

    # 5. Ticket group size
    df['TicketGroupSize'] = (
        df.groupby('Ticket')['Ticket'].transform('count')
    )

    return df

In [ ]:
train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

###Remove Unnecessary Columns

In [ ]:
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']

train_df = train_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

In [ ]:
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

print("\nTraining features:")
print(train_df.columns.tolist())

print("\nTesting features:")
print(test_df.columns.tolist())

# Data Preparation

In [ ]:
# Separate features and target
X = train_df.drop(columns=['Survived'])
y = train_df['Survived']

# Display dataset dimensions
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Test data shape:", test_df.shape)

In [ ]:
# Numerical features
numeric_features = [
    'Pclass', 'Age', 'SibSp', 'Parch',
    'Fare', 'FamilySize', 'IsAlone',
    'TicketGroupSize'
]

# Categorical features
categorical_features = [
    'Sex', 'Embarked', 'Title', 'Deck'
]

In [ ]:
print("Missing values in training features:")
print(X.isnull().sum()[X.isnull().sum() > 0])

print("\nMissing values in test features:")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])

###Train-Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X = train_df.drop('Survived', axis=1)
y = train_df['Survived']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

###Build the Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

###Combine the Preprocessing Pipelines

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Model Building & Initial Evaluation

##Logistic Regression

In [ ]:
# Model Definition
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [ ]:
# Train the Logistic Regression model
logistic_model.fit(X_train, y_train)

In [ ]:
# Predict survival on the validation set
y_pred_lr = logistic_model.predict(X_valid)

###Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_lr = accuracy_score(y_valid, y_pred_lr)

print(f"Logistic Regression Accuracy: {accuracy_lr:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_lr = confusion_matrix(y_valid, y_pred_lr)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lr,
    display_labels=['Not Survived', 'Survived']
)

disp.plot(cmap='Blues')

plt.title('Confusion Matrix - Logistic Regression')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_valid,
    y_pred_lr,
    target_names=['Not Survived', 'Survived']
))

###Logistic Regression Feature Importance

In [ ]:
# Get feature names after preprocessing
feature_names_lr = logistic_model.named_steps['preprocessor'].get_feature_names_out()

# Get Logistic Regression coefficients
coefficients_lr = logistic_model.named_steps['classifier'].coef_[0]

# Create a DataFrame
lr_coefficients = pd.DataFrame({
    'Feature': feature_names_lr,
    'Coefficient': coefficients_lr
})

# Calculate absolute coefficient values
lr_coefficients['Abs_Coefficient'] = lr_coefficients['Coefficient'].abs()

# Sort by absolute importance
lr_coefficients = lr_coefficients.sort_values(
    by='Abs_Coefficient',
    ascending=False
)

lr_coefficients

In [ ]:
plt.figure(figsize=(10, 7))

colors = [
    'green' if value > 0 else 'red'
    for value in lr_coefficients['Coefficient']
]

plt.barh(
    lr_coefficients['Feature'],
    lr_coefficients['Coefficient'],
    color=colors
)

plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coefficient')
plt.ylabel('Features')
plt.title('Logistic Regression - Feature Coefficients')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

##Random Forest

In [ ]:
# Model Definition

from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1
    ))
])

In [ ]:
# Train the Random Forest model
rf_model.fit(X_train, y_train)

In [ ]:
# Predict survival on the validation set
y_pred_rf = rf_model.predict(X_valid)

###Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_rf = accuracy_score(y_valid, y_pred_rf)

print("Random Forest Accuracy:", accuracy_rf)

print("\nClassification Report:")
print(classification_report(
    y_valid,
    y_pred_rf,
    target_names=['Not Survived', 'Survived']
))

###Random Forest Feature Importance

In [ ]:
# Get feature names after preprocessing
feature_names_rf = rf_model.named_steps['preprocessor'].get_feature_names_out()

# Get feature importances
importances_rf = rf_model.named_steps['classifier'].feature_importances_

# Create a DataFrame
rf_importance = pd.DataFrame({
    'Feature': feature_names_rf,
    'Importance': importances_rf
})

# Sort features by importance
rf_importance = rf_importance.sort_values(
    by='Importance',
    ascending=False
)

rf_importance

In [ ]:
plt.figure(figsize=(10, 8))

plt.barh(
    rf_importance['Feature'],
    rf_importance['Importance']
)

plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Random Forest - Feature Importance')

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## XGBoost

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [ ]:
# Train the XGBoost model
xgb_model.fit(X_train, y_train)

In [ ]:
# Predict survival on the validation set
y_pred_xgb = xgb_model.predict(X_valid)

###Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_xgb = accuracy_score(y_valid, y_pred_xgb)

print("XGBoost Accuracy:", accuracy_xgb)

print("\nClassification Report:")
print(classification_report(
    y_valid,
    y_pred_xgb,
    target_names=['Not Survived', 'Survived']
))

##Initial Model Comparison

In [ ]:
model_comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'XGBoost'
    ],
    'Accuracy': [
        accuracy_lr,
        accuracy_rf,
        accuracy_xgb
    ]
})

model_comparison = model_comparison.sort_values(
    by='Accuracy',
    ascending=False
)

model_comparison

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=model_comparison,
    x='Accuracy',
    y='Model'
)

plt.title('Initial Model Performance Comparison')
plt.xlabel('Accuracy')
plt.ylabel('Model')
plt.xlim(0, 1)

plt.tight_layout()
plt.show()

###XGBoost Feature Importance

In [ ]:
# Get feature names after preprocessing
feature_names_xgb = (
    xgb_model.named_steps['preprocessor'].get_feature_names_out()
)

# Get the trained XGBoost classifier
xgb_classifier = xgb_model.named_steps['classifier']

# Extract feature importances using Gain
importances_xgb = xgb_classifier.feature_importances_

# Create a DataFrame
xgb_importance = pd.DataFrame({
    'Feature': feature_names_xgb,
    'Importance': importances_xgb
})

# Sort by importance
xgb_importance = xgb_importance.sort_values(
    by='Importance',
    ascending=False
)

xgb_importance

In [ ]:
plt.figure(figsize=(10, 8))

plt.barh(
    xgb_importance['Feature'],
    xgb_importance['Importance']
)

plt.xlabel('Feature Importance (Gain)')
plt.ylabel('Features')
plt.title('XGBoost - Feature Importance')

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Model Evaluation Using Cross-Validation

###Configure Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

In [ ]:
# Configure Stratified K-Fold Cross-Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

###Logistic Regression – Cross-Validation

In [ ]:
# Evaluate Logistic Regression using cross-validation
cv_scores_lr = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Scores :", cv_scores_lr)
print("Mean Accuracy :", cv_scores_lr.mean())
print("Standard Deviation :", cv_scores_lr.std())

###Random Forest – Cross-Validation

In [ ]:
# Evaluate Random Forest using cross-validation
cv_scores_rf = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Scores :", cv_scores_rf)
print("Mean Accuracy :", cv_scores_rf.mean())
print("Standard Deviation :", cv_scores_rf.std())

###XGBoost – Cross-Validation

In [ ]:
# Evaluate XGBoost using cross-validation
cv_scores_xgb = cross_val_score(
    xgb_model,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Scores :", cv_scores_xgb)
print("Mean Accuracy :", cv_scores_xgb.mean())
print("Standard Deviation :", cv_scores_xgb.std())

In [ ]:
# Compare cross-validation results
cv_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Mean Accuracy': [
        cv_scores_lr.mean(),
        cv_scores_rf.mean(),
        cv_scores_xgb.mean()
    ],
    'Standard Deviation': [
        cv_scores_lr.std(),
        cv_scores_rf.std(),
        cv_scores_xgb.std()
    ]
})

cv_comparison = cv_comparison.sort_values(
    by='Mean Accuracy',
    ascending=False
)

cv_comparison

# XGBoost Hyperparameter Optimization



In [ ]:
# Define the hyperparameter search space
param_dist = {
    'classifier__n_estimators': [100, 200, 300, 400, 500],
    'classifier__max_depth': [2, 3, 4, 5, 6],
    'classifier__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'classifier__subsample': [0.7, 0.8, 0.9, 1.0],
    'classifier__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'classifier__min_child_weight': [1, 3, 5, 7],
    'classifier__gamma': [0, 0.1, 0.2, 0.3]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Configure RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=50,
    scoring='accuracy',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
# Display the best hyperparameters
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Mean Cross-Validation Accuracy:")
print(random_search.best_score_)

In [ ]:
# Retrieve the best fitted model
best_xgb_model = random_search.best_estimator_

###Nested Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Configure outer cross-validation
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Configure inner cross-validation
inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [ ]:
# Configure the inner hyperparameter search
random_search_inner = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=25,
    scoring='accuracy',
    cv=inner_cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [ ]:
from sklearn.model_selection import cross_validate

# Evaluate the hyperparameter optimization process
nested_cv_results = cross_validate(
    estimator=random_search_inner,
    X=X_train,
    y=y_train,
    cv=outer_cv,
    scoring='accuracy',
    n_jobs=1,
    return_train_score=False
)

# Extract outer fold scores
nested_scores = nested_cv_results['test_score']

print("Nested CV Scores:", nested_scores)
print("Mean Accuracy:", nested_scores.mean())
print("Standard Deviation:", nested_scores.std())

###Evaluation of the Optimized XGBoost Model

In [ ]:
# Predict survival probabilities on the validation set
y_valid_proba = best_xgb_model.predict_proba(X_valid)[:, 1]

###Threshold Optimization

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Define classification thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

# Evaluate different thresholds
for threshold in thresholds:

    y_pred_threshold = (y_valid_proba >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Accuracy:", round(accuracy_score(y_valid, y_pred_threshold), 4))
    print("Precision:", round(precision_score(y_valid, y_pred_threshold), 4))
    print("Recall:", round(recall_score(y_valid, y_pred_threshold), 4))
    print("F1-score:", round(f1_score(y_valid, y_pred_threshold), 4))

Confusion Matrix – Threshold = 0.4

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Apply the classification threshold
y_pred_04 = (y_valid_proba >= 0.4).astype(int)

# Compute the confusion matrix
cm = confusion_matrix(y_valid, y_pred_04)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Survived', 'Survived']
)

disp.plot(cmap='Blues')
plt.title('Confusion Matrix - XGBoost (Threshold = 0.4)')
plt.show()

###Model Performance Evaluation

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Calculate ROC curve values
fpr, tpr, thresholds_roc = roc_curve(y_valid, y_valid_proba)

# Calculate AUC
auc_score = roc_auc_score(y_valid, y_valid_proba)

# Plot the ROC curve
plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f'XGBoost (AUC = {auc_score:.4f})'
)

# Add random classifier reference line
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - XGBoost Optimized')
plt.legend()
plt.grid()
plt.show()

print("AUC:", auc_score)

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# Calculate Precision-Recall curve values
precision, recall, thresholds_pr = precision_recall_curve(
    y_valid,
    y_valid_proba
)

# Calculate Average Precision
ap_score = average_precision_score(y_valid, y_valid_proba)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))

plt.plot(
    recall,
    precision,
    label=f'XGBoost (AP = {ap_score:.4f})'
)

# Add baseline reference line
baseline = y_valid.mean()

plt.axhline(
    y=baseline,
    linestyle='--',
    label=f'Baseline = {baseline:.4f}'
)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - XGBoost Optimized')
plt.legend()
plt.grid()
plt.show()

print("Average Precision:", ap_score)

###Model Interpretability Using SHAP

In [ ]:
import shap

# Extract the fitted preprocessor and XGBoost classifier
preprocessor_fitted = best_xgb_model.named_steps['preprocessor']
xgb_classifier = best_xgb_model.named_steps['classifier']

# Transform the validation data
X_valid_transformed = preprocessor_fitted.transform(X_valid)

# Convert sparse matrix to dense array if necessary
if hasattr(X_valid_transformed, 'toarray'):
    X_valid_transformed = X_valid_transformed.toarray()

# Get feature names after preprocessing
feature_names_shap = preprocessor_fitted.get_feature_names_out()

print("Transformed data shape:", X_valid_transformed.shape)
print("Number of features:", len(feature_names_shap))

In [ ]:
# Create the SHAP explainer
explainer = shap.TreeExplainer(xgb_classifier)

# Calculate SHAP values
shap_values = explainer.shap_values(X_valid_transformed)

print("SHAP values shape:", np.array(shap_values).shape)

In [ ]:
# Display SHAP summary plot
shap.summary_plot(
    shap_values,
    X_valid_transformed,
    feature_names=feature_names_shap
)

In [ ]:
# Display SHAP feature importance
shap.summary_plot(
    shap_values,
    X_valid_transformed,
    feature_names=feature_names_shap,
    plot_type='bar'
)

# Final Model Training

###Verify Data Dimensions and Features

In [ ]:
# Check dataset dimensions
print("X:", X.shape)
print("y:", y.shape)
print("test_df:", test_df.shape)

# Check feature consistency
print("Features in X:", list(X.columns))
print("Features in test_df:", list(test_df.columns))

# Verify that both datasets have the same features
assert list(X.columns) == list(test_df.columns), "Feature mismatch between training and test data!"

In [ ]:
# Verify passenger identifiers
print("Number of Passenger IDs:", len(test_passenger_ids))
print(test_passenger_ids.head())

# Check for missing or duplicated identifiers
print("Missing IDs:", test_passenger_ids.isnull().sum())
print("Duplicated IDs:", test_passenger_ids.duplicated().sum())

###Train the Final Model

In [ ]:
# Train the final model on the complete training dataset
best_xgb_model.fit(X, y)

###Generate Test Predictions

In [ ]:
# Generate survival predictions
y_test_pred = best_xgb_model.predict(test_df)

# Check prediction results
print("Number of predictions:", len(y_test_pred))
print("Prediction distribution:")
print(pd.Series(y_test_pred).value_counts())

In [ ]:
# Create the submission DataFrame
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': y_test_pred
})

# Verify the submission
print(submission.head())
print("\nDimensions:", submission.shape)

# Check for missing values
print("\nMissing values:")
print(submission.isnull().sum())

In [ ]:
# Export the submission file
submission.to_csv('submission.csv', index=False)

print("submission.csv created successfully!")

In [ ]:
# Load and inspect the generated submission file
submission_check = pd.read_csv('submission.csv')

print(submission_check.head())
print("\nSubmission shape:", submission_check.shape)